# Figure 1: Seasonal hourly distributions

Show the seasonal distribution of hourly temperature, RH, PM2.5, and CO2 in all 23 homes. The PM2.5 axis is logarithmic for display; the observations remain on their original scale.

**Step:** 12. Article figure. See README.md for its data prerequisites.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Read the retained hourly CSV and the crosswalk; map historical source labels to manuscript Home 1 through Home 23.

**Outputs:** The named figure in the separate reproduction figures directory, using the original export formats.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure figure paths and resolution

Only presentation and output settings belong here. Keep the manuscript data and event specification fixed when reproducing the final figure.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

# Set a lower DPI only for an explicit draft; manuscript exports use 900.
FIGURE_DPI = int(os.environ.get("INDOOR_AQ_FIGURE_DPI", "900"))
os.environ["INDOOR_AQ_OUTPUT"] = str(output_root(OUTPUT))
os.environ["INDOOR_AQ_FIGURE_DPI"] = str(FIGURE_DPI)
# All event-result figures use this run's results folder.


## Load plotting libraries and define the figure settings

The following constants preserve the current figure's outcome order, labels, colors, panel geometry, and reference lines. Numerical methodology and interpretation are documented in the preceding and following cells.

In [ ]:
import os
import sys
from pathlib import Path
from shared import CODE, DATA, output_root, load_hourly as load_manuscript_hourly

import warnings

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, NullLocator, ScalarFormatter

# ---- Input data ------------------------------------------------------------
DATA_DIR = DATA
HOURLY_PATH = DATA_DIR / 'hourly_primary_23.csv.gz'
CROSSWALK_PATH = DATA / 'home_id_crosswalk_23.csv'

# ---- Output ----------------------------------------------------------------
OUTPUT_DIR = output_root() / 'figures'
OUTPUT_STEM = 'Fig1_seasonal_distributions'
EXPORT_FORMATS = ('png', 'pdf', 'svg')
DPI = int(os.environ.get('INDOOR_AQ_FIGURE_DPI', '900'))

# ---- Font and library versions used for the manuscript figure --------------
FONT_FAMILY = 'Liberation Sans'
FALLBACK_FONT = 'Arial'  # same character widths as Liberation Sans, so the layout is unchanged
# Folders searched for LiberationSans-*.ttf if the font is not installed.
FONT_SEARCH_DIRS = [
    Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts',
    Path(os.environ.get('LOCALAPPDATA', '')) / 'Microsoft' / 'Windows' / 'Fonts',
    Path(r'C:\Program Files\LibreOffice\share\fonts\truetype'),
    # Optional: a folder with LiberationSans-*.ttf (e.g. unzipped from the
    # Liberation fonts release) can be supplied through INDOOR_AQ_FONT_DIR.
    *([Path(os.environ['INDOOR_AQ_FONT_DIR'])] if os.environ.get('INDOOR_AQ_FONT_DIR') else []),
]
REQUIRED_PANDAS_MAJOR = 2     # portable CSV input, no pickle dependency
EXPECTED_MATPLOTLIB = '3.11'  # other releases may shift text by a sub-pixel

# ---- Panels (top to bottom) -------------------------------------------------
VARIABLES = ['temp', 'rh', 'pm25', 'co2']
PANEL_LETTERS = 'abcd'
Y_LABELS = {
    'temp': 'Temperature (°C)',
    'rh': 'Relative humidity (%)',
    'pm25': r'PM$_{2.5}$ ($\mu$g/m$^3$)',
    'co2': r'CO$_2$ (ppm)',
}

# ---- Seasons ----------------------------------------------------------------
# Seasons are already assigned in the hourly data:
# non-heating = April–October, heating = November–March.
SEASONS = ['Non-heating season', 'Heating season']
SEASON_COLORS = {'Non-heating season': '#92C6A0', 'Heating season': '#DE9F83'}

# ---- Guideline / suggested values (dashed lines) ----------------------------
REFERENCE_LINES = {
    'temp': [18, 26],  # °C, lower and upper suggested indoor temperatures
    'rh': [60],        # %, upper suggested relative humidity
    'pm25': [5],       # µg/m3, WHO annual guideline
    'co2': [1000],     # ppm, ventilation indicator
}
REFERENCE_COLOR = '#68645F'

# ---- Logarithmic PM2.5 axis -------------------------------------------------
PM25_TICKS = [1, 2, 5, 10, 25, 50, 100, 250, 500, 1000]
PM25_YLIM = (0.8, 1500)

# ---- Box geometry -------------------------------------------------------------
# Home i sits at x = i (0-22); its two seasonal boxes are centered at i ± 0.18.
BOX_SPACING = 0.36
BOX_WIDTH = 0.28
X_LIMITS = (-0.65, 22.65)

# ---- Figure layout (inches and figure fractions) -----------------------------
# Only the bottom panel carries the Home ID labels. Panel height and the top and
# bottom margins are those of the earlier 9.0-inch layout (subplot fractions
# bottom 0.064, top 0.932, hspace 0.47); the gap between panels is 0.30 inch.
PANEL_HEIGHT = 9.0 * (0.932 - 0.064) / (4 + 3 * 0.47)   # 1.444 in
TOP_MARGIN = 9.0 * (1 - 0.932)                           # 0.612 in: legend and panel letter
BOTTOM_MARGIN = 9.0 * 0.064                              # 0.576 in: Home ID labels
PANEL_GAP = 0.30                                         # in
FIGURE_HEIGHT = TOP_MARGIN + 4 * PANEL_HEIGHT + 3 * PANEL_GAP + BOTTOM_MARGIN   # 7.86 in
FIGURE_SIZE = (7.4, FIGURE_HEIGHT)
SUBPLOT_LAYOUT = dict(left=0.112, right=0.992, bottom=BOTTOM_MARGIN / FIGURE_HEIGHT,
                      top=1 - TOP_MARGIN / FIGURE_HEIGHT, hspace=PANEL_GAP / PANEL_HEIGHT)
LEGEND_ANCHOR = (0.54, 1 - 9.0 * (1 - 0.996) / FIGURE_HEIGHT)   # 0.036 in below the top edge

## Check environment

Check the data-library requirement and report a rendering-version difference. Reading CSV avoids the former pandas-specific pickle dependency. A version warning does not change the plotted observations.

In [ ]:
def check_environment():
    """Check library versions against those used for the manuscript figure.

    Raises
    ------
    ImportError
        If pandas is older than 2.0.

    Warns
    -----
    UserWarning
        If matplotlib is not 3.11.x. The plotted data are unaffected, but text
        can be placed a sub-pixel differently, so the PNG would not be
        pixel-identical to the manuscript version.
    """
    if int(pd.__version__.split('.')[0]) < REQUIRED_PANDAS_MAJOR:
        raise ImportError(
            f'pandas >= {REQUIRED_PANDAS_MAJOR}.0 is needed to read {HOURLY_PATH.name}; '
            f'this kernel has pandas {pd.__version__}.')
    if not matplotlib.__version__.startswith(EXPECTED_MATPLOTLIB + '.'):
        warnings.warn(
            f'The manuscript figure was rendered with matplotlib {EXPECTED_MATPLOTLIB}.x; '
            f'this kernel has {matplotlib.__version__}.')

## Apply style

Find the figure font (Liberation Sans, as in Figures 3, 4 and S1; Arial has the same character widths), then reset Matplotlib defaults before applying the figure's font, line, and export settings. This prevents unrelated notebook styling from carrying over into the manuscript figure.

In [ ]:
def resolve_font(family=FONT_FAMILY, search_dirs=FONT_SEARCH_DIRS, fallback=FALLBACK_FONT):
    """Return a font family that matplotlib can use, preferring ``family``.

    If ``family`` is not installed, any matching TrueType files found in
    ``search_dirs`` (e.g. LiberationSans-Regular.ttf) are registered for this
    session. If none are found, ``fallback`` is returned with a warning.
    """
    def available():
        return {font.name for font in font_manager.fontManager.ttflist}

    if family in available():
        return family

    pattern = family.replace(' ', '') + '-*.ttf'
    for directory in search_dirs:
        for path in sorted(Path(directory).glob(pattern)):
            font_manager.fontManager.addfont(str(path))

    if family in available():
        return family
    warnings.warn(f'{family} not found; using {fallback}. Glyph shapes will differ slightly.')
    return fallback


def apply_style(font_family):
    """Reset matplotlib to its defaults, then apply the Fig. 1 style.

    Resetting first keeps the output independent of earlier notebook cells
    and of any personal matplotlibrc file.
    """
    plt.rcdefaults()
    plt.rcParams.update({
        'font.family': font_family,
        'font.size': 9.5,
        'axes.labelsize': 10.5,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 0.65,
        'savefig.facecolor': 'white',
        'pdf.fonttype': 42,      # embed TrueType fonts so PDF text stays editable
        'svg.fonttype': 'none',  # keep SVG text as text rather than paths
    })

## Load hourly

Load hourly measurements through the shared sensor-based crosswalk. Missing observations stay missing; no additional filtering or concentration transformation is introduced here.

In [ ]:
def load_hourly(path=HOURLY_PATH):
    """Load the quality-checked hourly observations for the 23 homes.

    Returns
    -------
    pandas.DataFrame
        Long format, one row per home, variable and hour. Columns used here:
        ``home`` (source label, e.g. 'Home 25'), ``sensor`` (sensor ID),
        ``variable`` ('temp', 'rh', 'pm25' or 'co2'), ``season`` and
        ``value`` (NaN where the hour is missing or failed QA/QC).
    """
    return load_manuscript_hourly(os.environ.get('INDOOR_AQ_OUTPUT'), source_labels=True)

## Order homes

Resolve each source home label to the current manuscript ID and validate the sensor match. Plot all 23 homes in numeric manuscript order.

In [ ]:
def order_homes(hourly, crosswalk_path=CROSSWALK_PATH):
    """Put the homes in plotting order and map them to manuscript Home IDs.

    Source labels are not consecutive (for example, source 'Home 25' is
    Home 23 in the paper). Each source label is mapped through the crosswalk,
    and the sensor recorded in the hourly data must match the crosswalk.

    Returns
    -------
    homes : list of str
        Source home labels in ascending numeric order.
    home_ids : list of int
        The matching manuscript Home IDs, which must run 1-23.
    """
    crosswalk = pd.read_csv(crosswalk_path).set_index('source_home')
    homes = sorted(hourly['home'].unique(), key=lambda label: int(label.split()[-1]))

    home_ids = []
    for home in homes:
        sensor = int(hourly.loc[hourly['home'].eq(home), 'sensor'].iloc[0])
        expected_sensor = int(crosswalk.loc[home, 'sensor_id'])
        assert sensor == expected_sensor, f'Home/sensor mismatch: {home}, {sensor}'
        home_ids.append(int(crosswalk.loc[home, 'home_id']))

    assert home_ids == list(range(1, 24)), 'Manuscript Home IDs should run 1-23'
    return homes, home_ids

## Group values

Group nonmissing observations by home, outcome, and season. These original measurements provide the boxplot distributions.

In [ ]:
def group_values(hourly):
    """Index the hourly value series by (home, variable, season)."""
    return {key: group['value']
            for key, group in hourly.groupby(['home', 'variable', 'season'], sort=False)}

## Draw seasonal boxes

Place the two seasonal distributions side by side for each home using the original boxplot geometry. All home and season positions remain consistent across outcome panels.

In [ ]:
def draw_seasonal_boxes(ax, grouped, homes, variable):
    """Draw a non-heating and a heating box for every home.

    Boxes show the median and interquartile range of the hourly values.
    Whiskers reach the most extreme value within 1.5 IQR of the box
    (matplotlib's default) and points beyond them are not drawn. Missing
    hours are dropped before the statistics are calculated.
    """
    values, positions, colors = [], [], []
    for i, home in enumerate(homes):
        for j, season in enumerate(SEASONS):
            values.append(grouped[(home, variable, season)].dropna().to_numpy())
            positions.append(i + (j - 0.5) * BOX_SPACING)  # i - 0.18, i + 0.18
            colors.append(SEASON_COLORS[season])

    boxes = ax.boxplot(
        values, positions=positions, widths=BOX_WIDTH,
        patch_artist=True, showfliers=False, manage_ticks=False,
        medianprops={'color': 'black', 'lw': 0.65},
        boxprops={'lw': 0.55}, whiskerprops={'lw': 0.55}, capprops={'lw': 0.55})
    for box, color in zip(boxes['boxes'], colors):
        box.set_facecolor(color)

## Format panel

Apply the panel's labels, tick formatting, limits, and panel letter. Only the bottom panel shows the Home ID labels; the panels above keep aligned tick marks. Axis display choices do not alter the model or measurement values.

In [ ]:
def format_panel(ax, variable, letter, home_ids, show_x_labels=True):
    """Set the Home ID axis, the y label, the grid and the panel letter.

    With ``show_x_labels=False`` the Home ID tick labels and axis title are
    omitted and only the aligned tick marks remain (panels above the bottom one).
    """
    ax.set_xticks(range(len(home_ids)), [str(home_id) for home_id in home_ids])
    ax.set_xlim(*X_LIMITS)
    if show_x_labels:
        ax.set_xlabel('Home ID', labelpad=3)
    else:
        ax.tick_params(axis='x', labelbottom=False)
    ax.set_ylabel(Y_LABELS[variable], labelpad=8)

    ax.grid(axis='y', color='#dddddd', lw=0.5, alpha=0.8)
    ax.set_axisbelow(True)

    # Panel letter placed 40 pt left of and 4 pt above the axes' top-left corner
    ax.annotate(f'{letter})', xy=(0, 1), xycoords='axes fraction',
                xytext=(-40, 4), textcoords='offset points',
                fontsize=11, fontweight='bold', va='bottom')

## Add reference lines

Draw the manuscript's selected reference values at their stated positions. These lines are contextual display guides; they do not remove observations or change model eligibility.

In [ ]:
def add_reference_lines(ax, variable):
    """Draw the guideline/suggested values for ``variable`` as dashed lines."""
    for value in REFERENCE_LINES[variable]:
        ax.axhline(value, color=REFERENCE_COLOR, ls='--', lw=0.7)

## Set log pm25 axis

Use logarithmic spacing while labeling ticks in the original concentration units. This display transform is separate from the log1p model sensitivity.

In [ ]:
def set_log_pm25_axis(ax):
    """Use a log y axis with plain-number labels at fixed PM2.5 ticks."""
    ax.set_yscale('log')
    ax.yaxis.set_major_locator(FixedLocator(PM25_TICKS))
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.yaxis.set_minor_locator(NullLocator())
    ax.set_ylim(*PM25_YLIM)

## Add season legend

Use one shared seasonal legend to keep color interpretation consistent across panels.

In [ ]:
def add_season_legend(fig):
    """Add the shared season legend above panel (a)."""
    handles = [Patch(facecolor=SEASON_COLORS[season], edgecolor='black', lw=0.5, label=season)
               for season in SEASONS]
    fig.legend(handles=handles, loc='upper center', ncol=2, frameon=False,
               bbox_to_anchor=LEGEND_ANCHOR, fontsize=10)

## Build figure

Assemble the complete manuscript panel layout from the validated inputs and drawing helpers. This function contains presentation logic and does not fit a statistical model.

In [ ]:
def build_figure(hourly, homes, home_ids):
    """Build the four stacked panels of Fig. 1 (one shared Home ID axis) and return the figure.

    Parameters
    ----------
    hourly : pandas.DataFrame
        Output of :func:`load_hourly`.
    homes, home_ids : list
        Output of :func:`order_homes`.
    """
    fig, axes = plt.subplots(len(VARIABLES), 1, figsize=FIGURE_SIZE)
    fig.subplots_adjust(**SUBPLOT_LAYOUT)

    grouped = group_values(hourly)
    for ax, variable, letter in zip(axes, VARIABLES, PANEL_LETTERS):
        draw_seasonal_boxes(ax, grouped, homes, variable)
        format_panel(ax, variable, letter, home_ids, show_x_labels=(variable == VARIABLES[-1]))
        add_reference_lines(ax, variable)
        if variable == 'pm25':
            set_log_pm25_axis(ax)

    add_season_legend(fig)
    return fig

## Save figure

Export the figure to the original file formats. PNG uses the configured resolution (900 dpi by default); PDF and SVG, when included, preserve vector elements. The output directory is separate from the manuscript source figures.

In [ ]:
def save_figure(fig, output_dir=OUTPUT_DIR, stem=OUTPUT_STEM,
                formats=EXPORT_FORMATS, dpi=DPI):
    """Save ``fig`` once per format and return the written paths.

    The PNG is written at ``dpi``; PDF and SVG are vector formats.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in formats:
        path = output_dir / f'{stem}.{ext}'
        fig.savefig(path, dpi=dpi)
        paths.append(path)
    return paths

## Build and export the figure

Read the retained hourly CSV and the crosswalk; map historical source labels to manuscript Home 1 through Home 23. The cells above expose each processing and plotting step for inspection. The following execution writes the figure and prints its actual paths.

In [ ]:
check_environment()
apply_style(resolve_font())

hourly = load_hourly()
homes, home_ids = order_homes(hourly)

fig = build_figure(hourly, homes, home_ids)
for path in save_figure(fig):
    print(f'Saved {path}')

## Display a compact notebook preview

The preview uses a smaller PNG rendered from the same figure object. The saved publication PNG remains at the configured 900 dpi. Keep methodological detail in the caption when preparing the manuscript.

In [ ]:
import io
from IPython.display import Image, display
_preview = io.BytesIO()
fig.savefig(_preview, format="png", dpi=110, bbox_inches="tight")
display(Image(data=_preview.getvalue()))
plt.close(fig)